# Starter notebook

Works in both modes:
- **local** — runs on your machine, MLflow on localhost, DVC pulls from S3
- **cloud** — run this notebook on the EC2 VM or package it as a training script

MLflow and DVC are pre-configured by devenv — no setup needed.

In [ ]:
import os
import mlflow
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Both of these are set by devenv — no need to configure manually
print("MLflow tracking URI:", os.environ.get('MLFLOW_TRACKING_URI'))
print("DVC remote URL:     ", os.environ.get('DVC_REMOTE_URL'))
print("Infra mode:         ", os.environ.get('INFRA_MODE', 'local'))

## 1. Load data

DVC tracks your data. After `dvc pull` your data is in `data/`.

Replace the sample data below with your own dataset.

In [ ]:
# Replace with: pd.read_csv('data/train.csv')
from sklearn.datasets import load_iris
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name='target')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {len(X_train)} rows  |  Test: {len(X_test)} rows")
X.head()

## 2. Train with MLflow tracking

In [ ]:
# Hyperparameters — tweak these
PARAMS = {
    "n_estimators": 100,
    "max_depth": 5,
    "random_state": 42,
}

mlflow.set_experiment("starter")

with mlflow.start_run() as run:
    print(f"Run ID: {run.info.run_id}")

    # Log hyperparameters
    mlflow.log_params(PARAMS)

    # Train
    model = RandomForestClassifier(**PARAMS)
    model.fit(X_train, y_train)

    # Evaluate
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    # Log metrics
    mlflow.log_metric("accuracy", acc)

    # Log model — this is what deploy <run-id> will package
    mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        registered_model_name="starter-model",
    )

    print(f"Accuracy: {acc:.4f}")
    print(f"\nTo deploy this run:")
    print(f"  deploy {run.info.run_id}")

RUN_ID = run.info.run_id

In [ ]:
print(classification_report(y_test, y_pred, target_names=iris.target_names))

## 3. Deploy

From the terminal:

```sh
# local mode — serves on localhost:5001
deploy <run-id>

# cloud mode — packages model + src/inference.py → SageMaker endpoint
deploy <run-id>
```

Edit `src/inference.py` to match your model flavour and set in `devenv.nix`:
```nix
env.INFERENCE_SCRIPT = "src/inference.py";
```

In [ ]:
# Quick local test — call the model directly before deploying
sample = X_test.iloc[:3]
predictions = model.predict(sample)
print("Sample predictions:", [iris.target_names[p] for p in predictions])

## 4. Save data with DVC

After acquiring/transforming data, push it to S3 so the team and SageMaker jobs can access it.

In [ ]:
import subprocess, pathlib

# Save training data
pathlib.Path('data').mkdir(exist_ok=True)
X_train.assign(target=y_train.values).to_csv('data/train.csv', index=False)
X_test.assign(target=y_test.values).to_csv('data/test.csv', index=False)
print("Saved data/train.csv and data/test.csv")

# Track with DVC (run in terminal, or uncomment below)
# subprocess.run(['dvc', 'add', 'data/'], check=True)
# subprocess.run(['dvc', 'push'], check=True)
print("\nTo version and push: dvc add data/ && dvc push")